# Grading Report

## Grade: 65/100

| Function | Score | Status |
|----------|-------|--------|
| `conv2d_loops` | 8/30 | ~ 3 test(s) failed |
| `im2patches` | 30/30 | ✓ Correct |
| `conv2d_vectorized` | 27/40 | ~ 1 test(s) failed |

---

### `loops_correctness` — FAIL

**Result:** `Max diff 20.969124`

**Description:** Correctness against reference on random input

---

### `loops_nonsquare` — FAIL

**Result:** `Max diff 13.141825`

**Description:** Non-square image and kernel

---

### `loops_is_conv_not_xcorr` — FAIL

**Result:** `Implemented cross-correlation, not convolution — kernel must be flipped`

**Description:** Verifies this is convolution (flipped kernel), not cross-correlation

---

### `vec_matches_loops` — FAIL

**Result:** `Vectorized and loop outputs differ, max diff 8.060059`

**Description:** Vectorized output must match the student's own loop output

---

---
---

# Original Submission

*Everything below is the student's original work, unmodified.*

---

# HW 1 — From Scratch: 2D Discrete Convolution
**Modern Computer Vision** — Technion, Spring 2026

**Due:** April 26, 2026

## Submission Requirements

**IMPORTANT:** Submit this notebook **with all outputs** — run all cells before submitting. Notebooks without outputs will lose points. Make sure your `STUDENT_ID` is set correctly.

**Student ID:**

In [1]:
STUDENT_ID = "error_student"

In [2]:
import torch
import time

---
## Background

Recall the 2D discrete convolution:

$$
(f * k)[m, n] = \sum_{i} \sum_{j} f[i, j] \;\cdot\; k[m - i,\; n - j]
$$

Note: this is the **mathematical** definition of convolution (not cross-correlation). The kernel is **flipped** before sliding.

We work with 2D tensors throughout — no batch or channel dimensions.

---
## Part 1: For-loop convolution

Implement 2D convolution using explicit for-loops over the output spatial positions.

Use **valid** padding (output size = `H - kH + 1` × `W - kW + 1`).

In [3]:
def conv2d_loops(image, kernel):
    """2D convolution — BUG: no kernel flip (cross-correlation)."""
    H, W = image.shape
    kH, kW = kernel.shape
    H_out, W_out = H - kH + 1, W - kW + 1
    output = torch.zeros(H_out, W_out)
    for i in range(H_out):
        for j in range(W_out):
            output[i, j] = (image[i:i+kH, j:j+kW] * kernel).sum()
    return output

### Sanity check

In [ ]:
# Quick test: convolution with a delta should return the image (cropped)
torch.manual_seed(0)
img = torch.randn(8, 8)
delta = torch.zeros(3, 3)
delta[1, 1] = 1.0

out = conv2d_loops(img, delta)
assert out.shape == (6, 6), f"Expected (6,6), got {out.shape}"
assert torch.allclose(out, img[1:-1, 1:-1], atol=1e-6), "Delta test failed"
print("✓ Delta test passed")

[Grand test: skipped execution of this cell for speed]


In [ ]:
# Commutativity test: f * g == g * f (on the overlapping valid region)
torch.manual_seed(42)
a = torch.randn(12, 12)
b = torch.randn(5, 5)

out1 = conv2d_loops(a, b)
# For commutativity, conv(a,b) should equal conv(b,a) — but sizes differ.
# Instead, verify against a known manual computation:
# Single output pixel: out[0,0] = sum over (i,j) of a[i,j] * b[-i,-j] for valid indices
kernel_flipped = b.flip(0).flip(1)
expected_00 = (a[:5, :5] * kernel_flipped).sum()
assert torch.allclose(out1[0, 0], expected_00, atol=1e-5), "Manual pixel check failed"
print(f"✓ Manual pixel check passed")
print(f"  Output shape: {out1.shape}")

[Grand test: skipped execution of this cell for speed]


### Timing

Measure how your loop implementation scales with image size.

In [ ]:
kernel = torch.randn(5, 5)
sizes = [32, 64, 128, 256]

print("Loop convolution timings:")
print(f"{'Size':>8}  {'Time (ms)':>10}")
print("-" * 22)

loop_times = {}
for s in sizes:
    img = torch.randn(s, s)
    # warm up
    conv2d_loops(img, kernel)
    # time it
    t0 = time.time()
    conv2d_loops(img, kernel)
    elapsed = (time.time() - t0) * 1000
    loop_times[s] = elapsed
    print(f"{s:>6}x{s}  {elapsed:>8.1f} ms")

[Grand test: skipped execution of this cell for speed]


**Question:** When you double the image size, by roughly what factor does the runtime increase? Why?

*Your answer here.*

---
## Part 2: Vectorized convolution via im2patches

The for-loop implementation is slow because Python loops over every output pixel.

Think about what each output pixel computes: it's a **dot product** between a local patch of the image and the (flipped) kernel. If we could collect *all* such patches into a single matrix, we could compute *all* output pixels with one matrix operation.

### Step 1: `im2patches`

Write a function that extracts every `(kH, kW)` patch from the image and stacks them into a 2D matrix.

**Constraints:**
- Do **not** use `torch.unfold`, `F.unfold`, or any built-in patching/convolution ops.
- You *may* loop over the kernel dimensions (this is a small loop, `kH × kW` iterations).
- Use **slicing** and **reshape** to build the matrix.

**Target shape:** `(H_out * W_out, kH * kW)`

where `H_out = H - kH + 1` and `W_out = W - kW + 1`.

In [4]:
def im2patches(image, kH, kW):
    """Extract patches from image for vectorized conv."""
    H, W = image.shape
    H_out, W_out = H - kH + 1, W - kW + 1
    patches = torch.zeros(H_out * W_out, kH * kW)
    for i in range(H_out):
        for j in range(W_out):
            patches[i * W_out + j] = image[i:i+kH, j:j+kW].reshape(-1)
    return patches

### Sanity check

In [ ]:
torch.manual_seed(0)
img = torch.arange(16.).reshape(4, 4)
print("Image:")
print(img)

patches = im2patches(img, 3, 3)
print(f"\nPatches shape: {patches.shape}  (expected: (4, 9))")
assert patches.shape == (4, 9), f"Wrong shape: {patches.shape}"

# First patch should be top-left 3x3 block, flattened
expected_first = img[:3, :3].reshape(-1)
assert torch.allclose(patches[0], expected_first), "First patch mismatch"
print("✓ im2patches test passed")

[Grand test: skipped execution of this cell for speed]


### Step 2: Convolution via matrix multiplication

Now use `im2patches` to implement convolution as a single matrix-vector product.

Remember: this is **convolution**, not cross-correlation — the kernel must be flipped.

In [5]:
def conv2d_vectorized(image, kernel):
    """Vectorized 2D convolution using im2patches."""
    kH, kW = kernel.shape
    H_out = image.shape[0] - kH + 1
    W_out = image.shape[1] - kW + 1
    patches = im2patches(image, kH, kW)
    kernel_flipped = kernel.flip(0).flip(1).reshape(-1)
    result = patches @ kernel_flipped
    return result.reshape(H_out, W_out)

### Correctness check

Your vectorized version must match the loop version exactly.

In [ ]:
torch.manual_seed(7)
img = torch.randn(20, 20)
kernel = torch.randn(5, 5)

out_loop = conv2d_loops(img, kernel)
out_vec = conv2d_vectorized(img, kernel)

assert out_loop.shape == out_vec.shape, f"Shape mismatch: {out_loop.shape} vs {out_vec.shape}"
assert torch.allclose(out_loop, out_vec, atol=1e-5), "Results don't match!"
print("✓ Vectorized output matches loop output")

[Grand test: skipped execution of this cell for speed]


### Timing comparison

In [ ]:
kernel = torch.randn(5, 5)
sizes = [32, 64, 128, 256]

print("Vectorized convolution timings:")
print(f"{'Size':>8}  {'Loop (ms)':>10}  {'Vec (ms)':>10}  {'Speedup':>8}")
print("-" * 45)

for s in sizes:
    img = torch.randn(s, s)
    # warm up
    conv2d_vectorized(img, kernel)
    # time vectorized
    t0 = time.time()
    conv2d_vectorized(img, kernel)
    vec_time = (time.time() - t0) * 1000
    
    lt = loop_times.get(s, float('nan'))
    speedup = lt / vec_time if vec_time > 0 else float('inf')
    print(f"{s:>6}x{s}  {lt:>8.1f} ms  {vec_time:>8.1f} ms  {speedup:>7.1f}x")

[Grand test: skipped execution of this cell for speed]


**Question:** Why is the vectorized version faster, even though it does the same number of multiplications?

*Your answer here.*

---
*Modern Computer Vision — Technion — Spring 2026*